In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiversx/phase2-helpers/phase2_runner.py
/kaggle/input/datasets/achiversx/phase2-helpers/diagnose_boundaries.py
/kaggle/input/datasets/achiversx/phase2-helpers/phase2_three_experiments.py
/kaggle/input/datasets/achiversx/phase2-helpers/ner_validate_submission.py
/kaggle/input/datasets/achiversx/phase2-helpers/phase2_backbones.py
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/__huggingface_repos__.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/validation_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/split_manifest.json
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_data/train_windows.pt
/kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_ar

# Phase 2 Legal NER — End-to-End Training to Submission

This notebook runs a reproducible candidate pipeline using **only the Phase 1 checkpoints and Phase 2 data attached in Kaggle**. It is designed to be executed top-to-bottom in a Kaggle GPU session.

## Experiments included

1. **Arch 2 + BitFit** — 768-token windows, stride 512, shifted last window.
2. **Arch 1 + BitFit** — bias-only adaptation plus BIO classification head.
3. **Arch 3 + Houlsby adapters** — rank/bottleneck 16 on attention output and MLP `fc2` projections.
4. **Arch 1 hard-label LoRA control** — matched control for distillation.
5. **Arch 1 LoRA + multi-teacher KD** — teacher probabilities blended from fine-tuned Arch 3 and Arch 4.
6. **Arch 4 full-fine-tuned checkpoint** — reuse the existing Phase 2 checkpoint and generate predictions.
7. **Existing Arch 4 LORMA artifact** — its saved validation score is recorded for analysis. The final Arch 4 candidate is selected from the verified full-fine-tuning model because the saved validation score is higher; loading the LORMA adapter for inference is not guessed from a checkpoint format.

Every completed experiment is saved to `/kaggle/working/phase2_architecture_results/<Arch + technique>/`. Each inference experiment folder contains `prediction.json`, `submission.zip`, `results.json`, `experiment_result.json`, `metrics.json`, metadata, and the trainable checkpoint where available. The metrics-only LORMA archive has no prediction or submission ZIP. The ZIP contains only a root-level `prediction.json`. The bundle also includes the four selected architecture submissions, a comparison CSV, a manifest, and `/kaggle/working/submission.zip` for the best validation-scoring candidate. The existing LORMA checkpoint is archived with its metrics and source checkpoint; because this notebook does not reconstruct its inference wrapper, it does not invent a LORMA prediction ZIP.

### Important notes

- In Kaggle, attach `achiversx/phase2-helpers`, the Phase 2 data, and the Arch 1, Arch 2, and Arch 3 Phase 1 model datasets before running. Also attach either the Phase 1 Arch 4 checkpoint or the Arch 4 LORMA checkpoint containing `architecture_config` if the Phase 1 config is unavailable.
- Select a Kaggle GPU accelerator, restart the session, then use **Run All** from the first cell so the cached teachers and student experiments run in order.
- Arch 4 uses the existing Phase-2 full-fine-tuning checkpoint. It prefers the original Phase-1 config; if that checkpoint is absent, it attempts a strict reconstruction using the architecture_config saved in the Arch-4 LORMA checkpoint and verifies the saved validation F1 reproduces. If neither config source is valid, attach the original Phase-1 Arch-4 checkpoint.
- The gold-label oracle is run before training; it tests window reconstruction and BIO decoding independently of model quality.
- Results are selected by the custom validation evaluator, not the private official test labels. The private test set is used only to generate submission predictions.
- This notebook has been assembled from your provided runner and experiment details; it cannot be executed outside your Kaggle session with the attached inputs.

## 1. Setup, discover inputs, import helpers

In [2]:
import os, sys, gc, json, math, time, glob, shutil, zipfile, subprocess, random, inspect
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator before running this notebook."
DEVICE = "cuda"
INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
RUN_ID = time.strftime("%Y%m%d_%H%M%S") + f"_{int(time.time() * 1000) % 100000:05d}"
RUN_ROOT = WORK_DIR / "phase2_runs_end2end"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
# Stable, human-readable artifact folders: one folder per architecture + technique.
ARTIFACT_ROOT = WORK_DIR / "phase2_architecture_results"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_HISTORY = ARTIFACT_ROOT / "_history"
ARTIFACT_BUNDLE_PATH = WORK_DIR / "phase2_architecture_results.zip"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Locate the helper package by its real file rather than assuming a hard-coded owner path.
runner_matches = list(INPUT_DIR.rglob("phase2_runner.py"))
if len(runner_matches) != 1:
    raise RuntimeError(f"Expected one phase2_runner.py, found {len(runner_matches)}: {runner_matches}")
HELPERS_DIR = runner_matches[0].parent
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import phase2_runner as R
from phase2_backbones import load_backbone
from phase2_runner import RunConfig, run_experiment, load_data, find_one, evaluate, compare_runs

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Helpers:", HELPERS_DIR)
print("Run ID:", RUN_ID)

print("\nAttached input top-level directories:")
for p in sorted(INPUT_DIR.iterdir()):
    print(" ", p)


def find_unique_file(filename, required=True, contains=None):
    matches = [p for p in INPUT_DIR.rglob(filename) if (contains is None or contains in str(p))]
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {filename!r} under {INPUT_DIR}; attach the required dataset.")
        return None
    if len(matches) > 1:
        paths = "\n".join(map(str, matches))
        raise RuntimeError(f"Multiple files named {filename!r}; add a 'contains' restriction.\n{paths}")
    return matches[0]


def fresh_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def unique_run_name(base):
    return f"{base}_{RUN_ID}"

PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory (GiB): 14.56
Helpers: /kaggle/input/datasets/achiversx/phase2-helpers
Run ID: 20261011_073309_89638

Attached input top-level directories:
  /kaggle/input/datasets
  /kaggle/input/models


## 2. Load standard and BitFit data; run the gold-label oracle

In [3]:
# Same 1024-token windows for Arch 1/3/4 and KD.
data = load_data(max_len=1024, stride=768, last_window="pad")
# Preserve Arch 2's original windowing configuration for reproducibility.
data_bitfit = load_data(max_len=768, stride=512, last_window="shift")

assert data.max_len == 1024 and data.stride == 768 and data.last_window == "pad"
assert data_bitfit.max_len == 768 and data_bitfit.stride == 512 and data_bitfit.last_window == "shift"
assert len(data.train_docs) == 56 and len(data.val_docs) == 14
assert len(data.test_text) == 35, f"Expected 35 test documents, got {len(data.test_text)}"

print("Standard windows:", len(data.train.input_ids), "train /", len(data.val.input_ids), "validation")
print("BitFit windows:", len(data_bitfit.train.input_ids), "train /", len(data_bitfit.val.input_ids), "validation")
print("Train / validation / test docs:", len(data.train_docs), len(data.val_docs), len(data.test_text))
print("Entity labels:", data.entity_labels)
print("BIO classes:", len(data.id2tag))
assert len(data.id2tag) == 25, f"Expected 25 BIO classes, got {len(data.id2tag)}"

# Official data and label paths needed for the validator.
TEST_TEXT_PATH = find_unique_file("fold3-text.json")
LABELS_PATH = find_unique_file("labels.txt")
VALIDATOR_PATH = find_unique_file("ner_validate_submission.py")
print("Test text:", TEST_TEXT_PATH)
print("Labels:", LABELS_PATH)
print("Validator:", VALIDATOR_PATH)

# Gold-label oracle: windowing + decoding must reproduce high strict F1.
import diagnose_boundaries as DB
oracle_main = DB.oracle_check(data)
oracle_bitfit = DB.oracle_check(data_bitfit)

if oracle_bitfit["strict_macro_f1"] < 0.95:
    raise RuntimeError("BitFit windowing oracle is unexpectedly low; fix window/offset pipeline before training.")
if oracle_main["strict_macro_f1"] < 0.95:
    raise RuntimeError("Standard windowing oracle is unexpectedly low; fix window/offset pipeline before training.")

documents: train 56, validation 14, test 35 | windows: train 502, validation 105 (max_len 1024, stride 768, last_window 'pad')
documents: train 56, validation 14, test 35 | windows: train 738, validation 157 (max_len 768, stride 512, last_window 'shift')
Standard windows: 502 train / 105 validation
BitFit windows: 738 train / 157 validation
Train / validation / test docs: 56 14 35
Entity labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
BIO classes: 25
Test text: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER/Test/fold3-text.json
Labels: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/ner_data/Phase_2_LNER/Train/labels.txt
Validator: /kaggle/input/datasets/achiversx/phase2-helpers/ner_validate_submission.py
oracle (gold tags through windowing+decode): strict macro-F1 0.9825 | exact 1551/1585 gold spans
  APP        gold    91  exact    86  F1 0.945
  RESP       gold    74  exact 

## 3. Reusable training, BitFit, decoding and submission utilities

In [4]:
def fingerprint(modules, trainable_ids):
    """Exact snapshots of every parameter excluded from the optimizer."""
    out = {}
    for mn, module in modules.items():
        for pn, p in module.named_parameters():
            if id(p) not in trainable_ids:
                out[f"{mn}.{pn}"] = (float(p.detach().double().sum().cpu()), float(p.detach().abs().max().cpu()))
    return out


def check_frozen(before, modules, trainable_ids):
    after = fingerprint(modules, trainable_ids)
    changed = [k for k in before if before[k] != after[k]]
    assert before.keys() == after.keys() and not changed, f"Frozen tensors changed: {changed[:5]}"
    print(f"Frozen-weight check PASS: {len(before)} tensors unchanged.")


def trainable_report(modules, optimizer):
    optimizer_ids = {id(p) for g in optimizer.param_groups for p in g["params"]}
    stray = []
    for mn, module in modules.items():
        for pn, p in module.named_parameters():
            if p.requires_grad and id(p) not in optimizer_ids:
                stray.append(f"{mn}.{pn}")
    assert not stray, f"Trainable parameters absent from optimizer: {stray[:10]}"
    n = sum(p.numel() for g in optimizer.param_groups for p in g["params"])
    print("Trainable parameters:", f"{n:,}", "| LR groups:", [g["lr"] for g in optimizer.param_groups])
    return optimizer_ids


def select_used_biases(bb, head, ids):
    """BitFit probe: enable bias tensors + head only, then retain biases with a gradient path."""
    for p in bb.model.parameters():
        p.requires_grad_(False)
    for p in head.parameters():
        p.requires_grad_(True)
    candidates = []
    candidate_names = []
    for name, p in bb.model.named_parameters():
        if name.endswith(".bias"):
            p.requires_grad_(True)
            candidates.append(p)
            candidate_names.append(name)
    if not candidates:
        raise RuntimeError("No bias tensors found; BitFit cannot be constructed.")
    ids = ids[:1].to(DEVICE)
    # Only bias/head parameters require gradients, reducing memory versus enabling every weight.
    with torch.enable_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
        logits = head(bb.hidden(ids))
        scalar = logits.float().sum()
    grads = torch.autograd.grad(scalar, candidates, allow_unused=True)
    selected = [(n, p) for n, p, g in zip(candidate_names, candidates, grads) if g is not None]
    selected_names = {n for n, _ in selected}
    for p in bb.model.parameters():
        p.requires_grad_(False)
    selected_params = []
    for name, p in bb.model.named_parameters():
        if name in selected_names:
            p.requires_grad_(True)
            selected_params.append(p)
    for p in head.parameters():
        p.requires_grad_(True)
    if not selected_params:
        raise RuntimeError("BitFit gradient probe selected zero bias parameters.")
    print(f"BitFit selected {len(selected_params)} bias tensors / {sum(p.numel() for p in selected_params):,} parameters")
    print("Selected bias examples:", [n for n, _ in selected[:10]])
    return selected_params


def _normalise_prediction_span(span, text, doc_id, valid_labels):
    """Validate 3-field spans and 4-field spans with surface text, preserving either schema."""
    if not isinstance(span, (list, tuple)) or len(span) not in (3, 4):
        raise ValueError(f"Invalid span format: {doc_id}: {span}")
    start, end, label = span[:3]
    if not isinstance(start, int) or not isinstance(end, int) or not (0 <= start < end <= len(text)):
        raise ValueError(f"Invalid character offsets: {doc_id}: {span}; text length={len(text)}")
    if label not in valid_labels:
        raise ValueError(f"Invalid entity label: {doc_id}: {label}")
    if len(span) == 4 and (not isinstance(span[3], str) or span[3] != text[start:end]):
        raise ValueError(f"Span surface text does not match source text: {doc_id}: {span}")
    return [int(start), int(end), str(label)] + ([str(span[3])] if len(span) == 4 else [])


def save_test_submission(run_name, dataset, tag_fn):
    """Encode test documents and save decoded spans, preserving their 3- or 4-field schema."""
    pred = {}
    valid_labels = set(dataset.entity_labels)
    for doc_id, text in dataset.test_text.items():
        enc = R._encode_test(dataset, text)
        spans = R._spans_for(enc, tag_fn, dataset)
        pred[str(doc_id)] = [
            _normalise_prediction_span(span, text, str(doc_id), valid_labels)
            for span in spans
        ]

    expected_ids = {str(k) for k in dataset.test_text.keys()}
    assert set(pred) == expected_ids, "Generated prediction IDs differ from official test IDs."
    valid_labels = set(dataset.entity_labels)
    for doc_id, spans in pred.items():
        text = dataset.test_text[doc_id] if doc_id in dataset.test_text else dataset.test_text[next(k for k in dataset.test_text if str(k) == doc_id)]
        for span in spans:
            assert len(span) == 3
            s, e, label = span
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(text), (doc_id, span, len(text))
            assert label in valid_labels, (doc_id, label)

    out_dir = WORK_DIR / f"sub_{run_name}"
    out_dir.mkdir(parents=True, exist_ok=True)
    json_path = out_dir / "prediction.json"
    zip_path = out_dir / "submission.zip"
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as zf:
        assert zf.namelist() == ["prediction.json"], zf.namelist()
    print("Submission ZIP created:", zip_path)
    return str(zip_path)


def get_candidate_zip(result, run_name):
    path = result.get("submission_zip") if isinstance(result, dict) else None
    if path and Path(path).is_file():
        return str(path)
    fallback = WORK_DIR / f"sub_{run_name}" / "submission.zip"
    if fallback.is_file():
        return str(fallback)
    raise FileNotFoundError(f"No submission ZIP found for run {run_name}: {path}")


def get_custom_tag_fn(hidden_fn, head, tag2id_src=None):
    """Build a tag function and reorder output classes into current data.id2tag order if mappings differ."""
    src_map = tag2id_src or data.tag2id
    dest_tags = [data.id2tag[i] if not isinstance(data.id2tag, dict) else data.id2tag[i] for i in range(len(data.id2tag))]
    if set(src_map) != set(data.tag2id):
        raise ValueError("Teacher BIO labels do not match the current Phase 2 labels.")
    order = [int(src_map[tag]) for tag in dest_tags]
    @torch.inference_mode()
    def tag_fn(win):
        x = torch.tensor([win], dtype=torch.long, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = head(hidden_fn(x)).float()
        logits = logits[..., order]
        return logits.argmax(-1)[0].tolist()
    return tag_fn


def artifact_folder_label(arch, method):
    """Human-readable stable folder name for every architecture/technique result."""
    a = str(arch).lower().replace(" ", "")
    m = str(method).lower()
    if a == "arch1" and "bitfit" in m:
        return "Arch 1 + BitFit"
    if a == "arch1" and ("distill" in m or "kd" in m):
        return "Arch 1 + LoRA + KD (Arch 3 + Arch 4)"
    if a == "arch1" and "lora" in m:
        return "Arch 1 + LoRA (Hard-label Control)"
    if a == "arch2" and "bitfit" in m:
        return "Arch 2 + BitFit"
    if a == "arch2" and "lora" in m:
        return "Arch 2 + LoRA"
    if a == "arch3" and "houlsby" in m:
        return "Arch 3 + Houlsby"
    if a == "arch3" and "lora" in m:
        return "Arch 3 + LoRA"
    if a == "arch4" and "lorma" in m:
        return "Arch 4 + LORMA"
    if a == "arch4":
        return "Arch 4 + Full Fine-Tuning"
    return f"{str(arch).title()} + {str(method).strip()}"


def _metrics_from_result(result):
    if not isinstance(result, dict):
        return {}
    best = result.get("best")
    if isinstance(best, dict):
        return best
    keys = ("strict_macro_f1", "strict_macro_precision", "strict_macro_recall",
            "gold_entities", "predicted_entities", "exact_matches", "per_label")
    return {k: result[k] for k in keys if k in result}


def archive_experiment_artifacts(arch, method, run_name, result,
                                 submission_zip=None, source_checkpoint=None,
                                 result_json_path=None, copy_source_checkpoint=False,
                                 note=None):
    """Save each experiment into a folder with prediction.json + submission.zip when available.

    Also saves validation metrics, runner results/config, trainable weights, checkpoint reference,
    and an official-format-validator report. Existing folders from another run are preserved in _history.
    """
    label = artifact_folder_label(arch, method)
    dest = ARTIFACT_ROOT / label
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

    # Preserve the previous result before refreshing the stable human-readable folder.
    if dest.exists() and any(dest.iterdir()):
        old_meta_path = dest / "metadata.json"
        try:
            old_run = json.loads(old_meta_path.read_text(encoding="utf-8")).get("run_id", "previous_run")
        except Exception:
            old_run = "previous_run"
        if old_run != RUN_ID:
            hist_dest = ARTIFACT_HISTORY / str(old_run) / label
            hist_dest.parent.mkdir(parents=True, exist_ok=True)
            if not hist_dest.exists():
                shutil.copytree(dest, hist_dest)
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)

    # The run result JSON written by the supplied runner is kept unchanged.
    if result_json_path is None:
        candidate_path = RUN_ROOT / str(run_name) / "results.json"
        result_json_path = candidate_path if candidate_path.is_file() else None
    if result_json_path is not None and Path(result_json_path).is_file():
        shutil.copy2(result_json_path, dest / "results.json")
    else:
        (dest / "results.json").write_text(
            json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
        )

    # Save the full returned object separately from the runner's original JSON.
    (dest / "experiment_result.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )
    metrics = _metrics_from_result(result)
    (dest / "metrics.json").write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )

    # Archive a trainable checkpoint if the runner generated one. Base backbones remain referenced,
    # rather than being copied repeatedly (Arch 4 full checkpoint is hundreds of MB).
    trained_ckpt = RUN_ROOT / str(run_name) / "best_trainable.pt"
    if trained_ckpt.is_file():
        shutil.copy2(trained_ckpt, dest / "best_trainable.pt")

    checkpoint_ref = {
        "run_id": RUN_ID,
        "run_name": str(run_name),
        "source_checkpoint": str(source_checkpoint) if source_checkpoint else None,
        "trainable_checkpoint": str(trained_ckpt) if trained_ckpt.is_file() else None,
        "source_checkpoint_copied": False,
    }
    if source_checkpoint and Path(source_checkpoint).is_file():
        source_checkpoint = Path(source_checkpoint)
        checkpoint_ref["source_checkpoint_size_bytes"] = source_checkpoint.stat().st_size
        # Copy small PEFT checkpoints only. Do not duplicate the 768MB full fine-tuned backbone.
        if copy_source_checkpoint and source_checkpoint.stat().st_size <= 64 * 1024 * 1024:
            shutil.copy2(source_checkpoint, dest / "source_checkpoint.pt")
            checkpoint_ref["source_checkpoint_copied"] = True
    (dest / "checkpoint_reference.json").write_text(
        json.dumps(checkpoint_ref, indent=2), encoding="utf-8"
    )

    zip_source = submission_zip or (result.get("submission_zip") if isinstance(result, dict) else None)
    prediction_available = False
    zip_validation = {"status": "not_generated", "reason": "No inference ZIP supplied."}
    if zip_source:
        zip_source = Path(zip_source)
        if not zip_source.is_file():
            raise FileNotFoundError(f"Submission ZIP does not exist: {zip_source}")
        with zipfile.ZipFile(zip_source, "r") as zf:
            names = zf.namelist()
            if names != ["prediction.json"]:
                raise AssertionError(f"Submission ZIP must contain only root prediction.json; found {names}")
            raw_prediction = zf.read("prediction.json")
        payload = json.loads(raw_prediction.decode("utf-8"))
        # Validate test IDs, entity labels, and character boundaries against the attached public test text.
        if "data" in globals() and hasattr(data, "test_text"):
            text_by_id = {str(k): v for k, v in data.test_text.items()}
            expected_ids = set(text_by_id)
            actual_ids = set(map(str, payload.keys()))
            if actual_ids != expected_ids:
                raise AssertionError(
                    f"{label}: test ID mismatch; missing={len(expected_ids-actual_ids)}, extra={len(actual_ids-expected_ids)}"
                )
            valid_labels = set(data.entity_labels)
            for doc_id, spans in payload.items():
                text = text_by_id[str(doc_id)]
                if not isinstance(spans, list):
                    raise TypeError(f"Predictions for {doc_id} must be a list")
                for span in spans:
                    _normalise_prediction_span(span, text, str(doc_id), valid_labels)
        (dest / "prediction.json").write_bytes(raw_prediction)
        shutil.copy2(zip_source, dest / "submission.zip")
        prediction_available = True
        zip_validation = {
            "status": "passed_internal_checks",
            "source_zip": str(zip_source),
            "test_documents": len(payload),
            "predicted_spans": sum(len(v) for v in payload.values()),
            "zip_entries": names,
        }

        # Run the official format validator per experiment, where the inputs are available.
        if all(name in globals() for name in ("VALIDATOR_PATH", "TEST_TEXT_PATH", "LABELS_PATH")):
            env = {**os.environ, "PRED": str(dest / "submission.zip"),
                   "TEXT": str(TEST_TEXT_PATH), "LABELS": str(LABELS_PATH)}
            proc = subprocess.run(
                [sys.executable, str(VALIDATOR_PATH)], env=env,
                capture_output=True, text=True
            )
            official_report = {
                "returncode": int(proc.returncode),
                "passed": proc.returncode == 0,
                "stdout": proc.stdout,
                "stderr": proc.stderr[-6000:],
            }
            (dest / "official_format_validator.json").write_text(
                json.dumps(official_report, indent=2), encoding="utf-8"
            )
            zip_validation["official_validator_passed"] = proc.returncode == 0
            if proc.returncode != 0:
                print(f"[WARN] Official validator failed for {label}; see official_format_validator.json")

    metadata = {
        "run_id": RUN_ID,
        "architecture": str(arch),
        "technique": str(method),
        "folder_name": label,
        "run_name": str(run_name),
        "created_at_local": time.strftime("%Y-%m-%d %H:%M:%S"),
        "artifact_dir": str(dest),
        "prediction_available": prediction_available,
        "submission_zip": str(dest / "submission.zip") if prediction_available else None,
        "prediction_json": str(dest / "prediction.json") if prediction_available else None,
        "validation_metrics": metrics,
        "zip_validation": zip_validation,
        "note": note,
    }
    (dest / "metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )
    readme = [
        f"Experiment: {label}",
        f"Run ID: {RUN_ID}",
        f"Run name: {run_name}",
        f"Architecture: {arch}",
        f"Technique: {method}",
        "",
        "Files:",
        "- prediction.json: model predictions over the official test texts (when prediction_available=true).",
        "- submission.zip: contains only prediction.json at the ZIP root (when available).",
        "- results.json: original Phase 2 runner result when available.",
        "- experiment_result.json: complete result object recorded by this notebook.",
        "- metrics.json: validation metrics.",
        "- best_trainable.pt: saved trainable state for trained PEFT candidates, when available.",
        "- checkpoint_reference.json: source/checkpoint paths and copy status.",
        "- official_format_validator.json: official format validator output, when executed.",
    ]
    if note:
        readme.extend(["", "Note:", str(note)])
    (dest / "README.txt").write_text("\n".join(readme) + "\n", encoding="utf-8")

    print(f"Archived {label}: {dest}")
    if prediction_available:
        print(f"  prediction.json + submission.zip ({zip_validation.get('predicted_spans')} spans)")
    else:
        print("  Metrics/checkpoint archived; prediction ZIP not generated for this artifact.")
    return dest


def run_candidate(name_base, arch, method, modules, hidden_fn, head, optimizer, dataset, epochs, notes, loss_fn=None, predict_test=True, scheduler=None):
    name = unique_run_name(name_base)
    cfg = RunConfig(
        name=name, arch=arch, method=method, epochs=epochs,
        batch_size=1, grad_accum=8, max_grad_norm=1.0, seed=SEED,
        amp=True, notes=notes, out_root=str(RUN_ROOT), overwrite=False,
    )
    trainable_ids = trainable_report(modules, optimizer)
    before = fingerprint(modules, trainable_ids)
    result = run_experiment(
        cfg=cfg, modules=modules, hidden_fn=hidden_fn, head=head,
        optimizer=optimizer, data=dataset, scheduler=scheduler,
        loss_fn=loss_fn, predict_test=predict_test, device=DEVICE,
    )
    check_frozen(before, modules, trainable_ids)
    result["submission_zip"] = get_candidate_zip(result, name) if predict_test else None
    artifact_dir = archive_experiment_artifacts(
        arch=arch, method=method, run_name=name, result=result,
        submission_zip=result["submission_zip"],
    )
    result["artifact_dir"] = str(artifact_dir)
    return name, result


## 4. Experiment A — Arch 2 + BitFit

In [5]:
print("=" * 78, "\nARCH 2 + BITFIT\n", "=" * 78)
bb2 = load_backbone("arch2", device=DEVICE)
head2 = nn.Linear(bb2.d_model, 25, device=next(bb2.model.parameters()).device, dtype=next(bb2.model.parameters()).dtype)
bitfit_params2 = select_used_biases(bb2, head2, data_bitfit.train.input_ids)
optimizer2 = torch.optim.AdamW(
    [{"params": bitfit_params2, "lr": 5e-4}, {"params": list(head2.parameters()), "lr": 2e-4}],
    weight_decay=0.01,
)
modules2 = {"backbone": bb2.model, "head": head2}
name2, result2 = run_candidate(
    "arch2_bitfit", "arch2", "BitFit (used biases + BIO head)", modules2,
    bb2.hidden, head2, optimizer2, data_bitfit, epochs=5,
    notes="Phase-1 Arch 2; BitFit biases selected by gradient probe; max_len=768 stride=512 last_window=shift; CE objective.",
)
print("Arch 2 BitFit best F1:", result2["best"]["strict_macro_f1"])
print("Arch 2 submission:", result2["submission_zip"])
del bb2, head2, bitfit_params2, optimizer2, modules2
fresh_gpu()

ARCH 2 + BITFIT
BitFit selected 48 bias tensors / 50,688 parameters
Selected bias examples: ['encoder.layers.0.norm1.bias', 'encoder.layers.0.self_attn.q_proj.bias', 'encoder.layers.0.self_attn.k_proj.bias', 'encoder.layers.0.self_attn.v_proj.bias', 'encoder.layers.0.self_attn.out_proj.bias', 'encoder.layers.0.norm2.bias', 'encoder.layers.0.ffn.linear1.bias', 'encoder.layers.0.ffn.linear2.bias', 'encoder.layers.1.norm1.bias', 'encoder.layers.1.self_attn.q_proj.bias']
Trainable parameters: 69,913 | LR groups: [0.0005, 0.0002]
[arch2_bitfit_20261011_073309_89638] trainable 69,913 = 0.054% of backbone (129,957,888); all modules 129,977,113
  epoch 1/5 batch 1/738 loss 3.3530
  epoch 1/5 batch 50/738 loss 2.7255
  epoch 1/5 batch 100/738 loss 1.8910
  epoch 1/5 batch 150/738 loss 1.6452
  epoch 1/5 batch 200/738 loss 0.8716
  epoch 1/5 batch 250/738 loss 0.4568
  epoch 1/5 batch 300/738 loss 1.5218
  epoch 1/5 batch 350/738 loss 0.3943
  epoch 1/5 batch 400/738 loss 0.2998
  epoch 1/5 batc

## 5. Experiment B — Arch 1 + BitFit

In [6]:
print("=" * 78, "\nARCH 1 + BITFIT\n", "=" * 78)
bb1_bit = load_backbone("arch1", device=DEVICE)
head1_bit = nn.Linear(bb1_bit.d_model, 25, device=next(bb1_bit.model.parameters()).device, dtype=next(bb1_bit.model.parameters()).dtype)
bitfit_params1 = select_used_biases(bb1_bit, head1_bit, data.train.input_ids)
optimizer1_bit = torch.optim.AdamW(
    [{"params": bitfit_params1, "lr": 5e-4}, {"params": list(head1_bit.parameters()), "lr": 2e-4}],
    weight_decay=0.01,
)
modules1_bit = {"backbone": bb1_bit.model, "head": head1_bit}
name1_bit, result1_bit = run_candidate(
    "arch1_bitfit", "arch1", "BitFit (used biases + BIO head)", modules1_bit,
    bb1_bit.hidden, head1_bit, optimizer1_bit, data, epochs=5,
    notes="Phase-1 Arch 1; BitFit biases selected by gradient probe; max_len=1024 stride=768 last_window=pad; CE objective.",
)
print("Arch 1 BitFit best F1:", result1_bit["best"]["strict_macro_f1"])
print("Arch 1 BitFit submission:", result1_bit["submission_zip"])
del bb1_bit, head1_bit, bitfit_params1, optimizer1_bit, modules1_bit
fresh_gpu()

ARCH 1 + BITFIT
BitFit selected 65 bias tensors / 68,352 parameters
Selected bias examples: ['blocks.0.norm1.bias', 'blocks.0.attn.q_proj.bias', 'blocks.0.attn.k_proj.bias', 'blocks.0.attn.v_proj.bias', 'blocks.0.attn.out_proj.bias', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc2.bias', 'blocks.1.norm1.bias', 'blocks.1.attn.q_proj.bias']
Trainable parameters: 87,577 | LR groups: [0.0005, 0.0002]
[arch1_bitfit_20261011_073309_89638] trainable 87,577 = 0.099% of backbone (88,210,944); all modules 88,230,169
  epoch 1/5 batch 1/502 loss 3.3255
  epoch 1/5 batch 50/502 loss 2.0926
  epoch 1/5 batch 100/502 loss 1.1282
  epoch 1/5 batch 150/502 loss 0.7675
  epoch 1/5 batch 200/502 loss 0.6303
  epoch 1/5 batch 250/502 loss 1.0615
  epoch 1/5 batch 300/502 loss 0.9639
  epoch 1/5 batch 350/502 loss 0.2795
  epoch 1/5 batch 400/502 loss 0.2978
  epoch 1/5 batch 450/502 loss 0.6032
  epoch 1/5 batch 500/502 loss 0.3428
  epoch 1/5 batch 502/502 loss 0.3494
[arch1_bitfit_202

## 6. Experiment C — Arch 3 + Houlsby adapters; cache its teacher logits

In [7]:
class HoulsbyAfterLinear(nn.Module):
    """Frozen linear projection with a zero-initialized residual bottleneck adapter."""
    def __init__(self, base_layer, d_model, bottleneck=16):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError(f"Expected nn.Linear, got {type(base_layer).__name__}")
        if base_layer.out_features != d_model:
            raise ValueError(f"Projection output must be {d_model}, got {base_layer.out_features}")
        self.base = base_layer
        device, dtype = base_layer.weight.device, base_layer.weight.dtype
        self.adapter_down = nn.Linear(d_model, bottleneck, device=device, dtype=dtype)
        self.adapter_act = nn.GELU()
        self.adapter_up = nn.Linear(bottleneck, d_model, device=device, dtype=dtype)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.adapter_up(self.adapter_act(self.adapter_down(y)))


def get_parent_module(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


print("=" * 78, "\nARCH 3 + HOULSBY\n", "=" * 78)
bb3 = load_backbone("arch3", device=DEVICE)
for p in bb3.model.parameters():
    p.requires_grad_(False)
probe = data.train.input_ids[:1].to(DEVICE)
with torch.inference_mode():
    hidden_before = bb3.hidden(probe).float().cpu()

wrapped = []
for i in range(len(bb3.model.blocks)):
    for suffix in ("attn.out_proj", "mlp.fc2"):
        name = f"blocks.{i}.{suffix}"
        parent, attr = get_parent_module(bb3.model, name)
        base = getattr(parent, attr)
        assert isinstance(base, nn.Linear) and base.out_features == bb3.d_model, (name, base)
        setattr(parent, attr, HoulsbyAfterLinear(base, bb3.d_model, bottleneck=16))
        wrapped.append(name)
assert len(wrapped) == 32

for module in bb3.model.modules():
    if isinstance(module, HoulsbyAfterLinear):
        for p in module.adapter_down.parameters(): p.requires_grad_(True)
        for p in module.adapter_up.parameters(): p.requires_grad_(True)
        for p in module.base.parameters(): p.requires_grad_(False)
head3 = nn.Linear(bb3.d_model, 25, device=next(bb3.model.parameters()).device, dtype=next(bb3.model.parameters()).dtype)
adapter_params3 = [p for p in bb3.model.parameters() if p.requires_grad]
optimizer3 = torch.optim.AdamW(
    [{"params": adapter_params3, "lr": 1e-4}, {"params": list(head3.parameters()), "lr": 2e-4}],
    weight_decay=0.01,
)
with torch.inference_mode():
    hidden_after = bb3.hidden(probe).float().cpu()
diff = (hidden_before - hidden_after).abs().max().item()
print("Adapters:", len(wrapped), "| trainable adapters:", f"{sum(p.numel() for p in adapter_params3):,}", "| identity max diff:", diff)
assert diff < 1e-5, "Houlsby adapters must preserve initial backbone outputs."

modules3 = {"backbone": bb3.model, "head": head3}
name3, result3 = run_candidate(
    "arch3_houlsby_r16", "arch3", "Houlsby adapters bottleneck=16", modules3,
    bb3.hidden, head3, optimizer3, data, epochs=5,
    notes="Phase-1 Arch 3; Houlsby-style adapters after attn.out_proj and mlp.fc2 in all blocks; bottleneck 16; CE objective.",
)
print("Arch 3 Houlsby best F1:", result3["best"]["strict_macro_f1"])
print("Arch 3 submission:", result3["submission_zip"])

@torch.inference_mode()
def cache_logits(hidden_fn, head, dataset, label_source_map=None, label="teacher"):
    N, T = dataset.train.input_ids.shape
    output = torch.empty((N, T, 25), dtype=torch.float16, device="cpu")
    # Reorder source head columns into this dataset's BIO ID order when a source map is supplied.
    if label_source_map is not None:
        tags = [dataset.id2tag[i] if not isinstance(dataset.id2tag, dict) else dataset.id2tag[i] for i in range(25)]
        if set(label_source_map) != set(dataset.tag2id):
            raise ValueError(f"{label} has a different BIO label vocabulary.")
        col_order = [int(label_source_map[tag]) for tag in tags]
    else:
        col_order = list(range(25))
    for i in range(N):
        ids = dataset.train.input_ids[i:i+1].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = head(hidden_fn(ids)).float()
        logits = logits[..., col_order]
        output[i] = logits[0].half().cpu()
        if i == 0 or (i + 1) % 100 == 0 or i + 1 == N:
            print(f"{label} logits: {i+1}/{N}")
    return output

teacher_cache_arch3 = cache_logits(bb3.hidden, head3, data, label="Arch 3 Houlsby")
print("Arch 3 teacher-cache shape:", tuple(teacher_cache_arch3.shape))

# Keep result metrics/ZIPs, release Arch 3 GPU model before loading Arch 4.
del bb3, head3, adapter_params3, optimizer3, modules3, hidden_before, hidden_after
fresh_gpu()

ARCH 3 + HOULSBY
Adapters: 32 | trainable adapters: 1,081,856 | identity max diff: 0.0
Trainable parameters: 1,107,481 | LR groups: [0.0001, 0.0002]
[arch3_houlsby_r16_20261011_073309_89638] trainable 1,107,481 = 0.455% of backbone (243,583,488); all modules 243,609,113
  epoch 1/5 batch 1/502 loss 3.4303
  epoch 1/5 batch 50/502 loss 2.7967
  epoch 1/5 batch 100/502 loss 1.8232
  epoch 1/5 batch 150/502 loss 0.8162
  epoch 1/5 batch 200/502 loss 0.6023
  epoch 1/5 batch 250/502 loss 1.0366
  epoch 1/5 batch 300/502 loss 0.9707
  epoch 1/5 batch 350/502 loss 0.2328
  epoch 1/5 batch 400/502 loss 0.2313
  epoch 1/5 batch 450/502 loss 0.4187
  epoch 1/5 batch 500/502 loss 0.1712
  epoch 1/5 batch 502/502 loss 0.2551
[arch3_houlsby_r16_20261011_073309_89638] epoch 1: loss 0.9745 | P 0.0006 R 0.0012 F1 0.0008  (custom validation)
  new best saved
  epoch 2/5 batch 1/502 loss 0.0685
  epoch 2/5 batch 50/502 loss 0.4350
  epoch 2/5 batch 100/502 loss 0.3002
  epoch 2/5 batch 150/502 loss 0.3

## 7. Reconstruct Arch 4 full-fine-tuned teacher from the provided checkpoint

In [8]:
def rotate_half(x):
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, dim, 2, dtype=torch.float32) / dim
            )
        )

        frequencies = torch.outer(
            torch.arange(max_seq_len, dtype=torch.float32),
            inv_freq
        )

        embeddings = torch.cat(
            [frequencies, frequencies],
            dim=-1
        )

        self.register_buffer(
            "cos",
            embeddings.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin",
            embeddings.sin(),
            persistent=False
        )

    def forward(self, T):
        return (
            self.cos[:T][None, None],
            self.sin[:T][None, None]
        )


# ------------------------------------------------------------
# 2. SwiGLU feed-forward network
# ------------------------------------------------------------

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.in_proj = nn.Linear(
            d_model,
            2 * d_ff,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        gate, value = self.in_proj(x).chunk(2, dim=-1)

        return self.out_proj(
            F.silu(gate) * value
        )


# ------------------------------------------------------------
# 3. Grouped-query attention
# ------------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_q,
        n_kv,
        window,
        is_global,
        dropout=0.0
    ):
        super().__init__()

        assert d_model % n_q == 0
        assert n_q % n_kv == 0

        self.n_q = n_q
        self.n_kv = n_kv

        self.head_dim = d_model // n_q
        self.rep = n_q // n_kv

        self.window = window
        self.is_global = is_global
        self.dropout = dropout

        q_dim = n_q * self.head_dim
        kv_dim = n_kv * self.head_dim

        self.split = [q_dim, kv_dim, kv_dim]

        self.qkv_proj = nn.Linear(
            d_model,
            q_dim + 2 * kv_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

    def forward(self, x, cos, sin, mask):
        B, T, C = x.shape

        Hq = self.n_q
        Hkv = self.n_kv
        Dh = self.head_dim
        rep = self.rep

        q, k, v = torch.split(
            self.qkv_proj(x),
            self.split,
            dim=-1
        )

        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)

        cos = cos.to(q.dtype)
        sin = sin.to(q.dtype)

        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin

        dropout_p = self.dropout if self.training else 0.0

        # Global causal attention
        if self.is_global:
            k = (
                k[:, :, None]
                .expand(B, Hkv, rep, T, Dh)
                .reshape(B, Hq, T, Dh)
            )

            v = (
                v[:, :, None]
                .expand(B, Hkv, rep, T, Dh)
                .reshape(B, Hq, T, Dh)
            )

            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                dropout_p=dropout_p,
                is_causal=True
            )

            return self.out_proj(
                y.transpose(1, 2).reshape(B, T, C)
            )

        # Local chunked attention
        W = self.window
        N = T // W

        qc = (
            q.view(B, Hq, N, W, Dh)
            .transpose(1, 2)
            .reshape(B * N, Hq, W, Dh)
        )

        def chunk_kv(t):
            tc = t.view(B, Hkv, N, W, Dh)

            previous = F.pad(
                tc[:, :, :-1],
                (0, 0, 0, 0, 1, 0)
            )

            combined = torch.cat(
                [previous, tc],
                dim=3
            )

            combined = combined.transpose(1, 2)[:, :, :, None]

            return (
                combined
                .expand(B, N, Hkv, rep, 2 * W, Dh)
                .reshape(B * N, Hq, 2 * W, Dh)
            )

        y = F.scaled_dot_product_attention(
            qc,
            chunk_kv(k),
            chunk_kv(v),
            attn_mask=mask,
            dropout_p=dropout_p
        )

        y = (
            y.view(B, N, Hq, W, Dh)
            .permute(0, 1, 3, 2, 4)
            .reshape(B, T, C)
        )

        return self.out_proj(y)


# ------------------------------------------------------------
# 4. Transformer block
# ------------------------------------------------------------

class Arch4Block(nn.Module):
    def __init__(self, cfg, idx):
        super().__init__()

        self.is_global = (
            (idx + 1) % cfg["GLOBAL_EVERY"] == 0
        )

        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])

        self.attn = GQAAttention(
            cfg["D_MODEL"],
            cfg["N_Q_HEADS"],
            cfg["N_KV_HEADS"],
            cfg["WINDOW"],
            self.is_global,
            cfg["DROPOUT"]
        )

        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])

        self.ffn = SwiGLU(
            cfg["D_MODEL"],
            cfg["D_FF"]
        )

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(
            self.norm1(x),
            cos,
            sin,
            mask
        )

        return x + self.ffn(
            self.norm2(x)
        )


# ------------------------------------------------------------
# 5. Original Arch4 backbone
# ------------------------------------------------------------

class Architecture4(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        W = cfg["WINDOW"]
        self.window = W

        assert cfg["MAX_SEQ_LEN"] % W == 0
        assert cfg["SEQ_LEN"] % W == 0

        self.token_embedding = nn.Embedding(
            cfg["VOCAB_SIZE"],
            cfg["D_MODEL"]
        )

        self.layers = nn.ModuleList([
            Arch4Block(cfg, i)
            for i in range(cfg["N_LAYERS"])
        ])

        self.final_norm = nn.LayerNorm(
            cfg["D_MODEL"]
        )

        # Phase-1 language-model head.
        # Its weight is tied to token_embedding.weight.
        self.lm_head = nn.Linear(
            cfg["D_MODEL"],
            cfg["VOCAB_SIZE"],
            bias=False
        )

        self.lm_head.weight = self.token_embedding.weight

        self.rope = RotaryEmbedding(
            cfg["D_MODEL"] // cfg["N_Q_HEADS"],
            cfg["MAX_SEQ_LEN"],
            cfg["ROPE_BASE"]
        )

        # Chunked local-attention mask
        n_chunks = cfg["MAX_SEQ_LEN"] // W

        i = torch.arange(W)[:, None]
        j = torch.arange(2 * W)[None, :]

        mask = (j > i) & (j <= i + W)
        mask = mask[None].repeat(n_chunks, 1, 1)

        mask[0] &= (j >= W)

        self.register_buffer(
            "chunk_mask",
            mask[:, None],
            persistent=False
        )

        self._init_weights()

    def _init_weights(self):
        std = self.cfg["INIT_STD"]

        nn.init.normal_(
            self.token_embedding.weight,
            0.0,
            std
        )

        for module in self.modules():
            if (
                isinstance(module, nn.Linear)
                and module.weight is not self.token_embedding.weight
            ):
                nn.init.normal_(
                    module.weight,
                    0.0,
                    std
                )

        scale = 1.0 / math.sqrt(
            2 * self.cfg["N_LAYERS"]
        )

        for block in self.layers:
            block.attn.out_proj.weight.data.mul_(scale)
            block.ffn.out_proj.weight.data.mul_(scale)

    def hidden_states(self, input_ids):
        B, T = input_ids.shape
        W = self.window

        assert T % W == 0

        N = T // W

        x = self.token_embedding(input_ids)

        cos, sin = self.rope(T)

        mask = (
            self.chunk_mask[:N][None]
            .expand(B, N, 1, W, 2 * W)
            .reshape(B * N, 1, W, 2 * W)
        )

        for block in self.layers:
            x = block(x, cos, sin, mask)

        return self.final_norm(x)

    def forward(self, input_ids):
        return self.lm_head(
            self.hidden_states(input_ids)
        )


# Architecture definition matches the supplied Arch 4 family.
# Prefer the original Phase-1 checkpoint's cfg. If that input is missing,
# recover the architecture config from the saved Arch-4 LORMA checkpoint,
# then load the full-fine-tuned Phase-2 backbone strictly and verify its
# validation score reproduces below. Never infer dimensions from tensor shapes.
ARCH4_REQUIRED_CFG_KEYS = (
    "VOCAB_SIZE", "D_MODEL", "N_LAYERS", "N_Q_HEADS", "N_KV_HEADS",
    "D_FF", "WINDOW", "GLOBAL_EVERY", "DROPOUT", "MAX_SEQ_LEN",
    "SEQ_LEN", "ROPE_BASE", "INIT_STD",
)


def find_phase1_arch4_checkpoint_optional():
    matches = [
        p for p in INPUT_DIR.rglob("best.pt")
        if "arch4_hybrid_gqa_201m" in str(p)
    ]
    if len(matches) > 1:
        raise RuntimeError(
            "Multiple Phase-1 Arch4 checkpoints found; narrow attached inputs:\n"
            + "\n".join(map(str, matches))
        )
    return matches[0] if matches else None


def find_arch4_fullft_checkpoint():
    # Restrict search to the Phase-2 baseline directory to avoid confusing it with other weights.
    return find_unique_file("best_strict_macro_f1.pt", contains="phase2_arch4_baseline")


def resolve_arch4_config():
    p1_path = find_phase1_arch4_checkpoint_optional()
    if p1_path is not None:
        p1 = torch.load(p1_path, map_location="cpu", weights_only=False)
        if not isinstance(p1, dict) or not isinstance(p1.get("cfg"), dict):
            raise RuntimeError(f"Phase-1 Arch4 checkpoint has no dict 'cfg': {p1_path}")
        cfg = dict(p1["cfg"])
        del p1
        source = f"Phase-1 checkpoint config: {p1_path}"
    else:
        lorma_path = find_unique_file(
            "best_lorma_strict_macro_f1.pt",
            contains="phase2_arch4_lorma",
            required=False,
        )
        if lorma_path is None:
            raise FileNotFoundError(
                "Cannot reconstruct Arch4: no Phase-1 checkpoint "
                "'arch4_hybrid_gqa_201m/best.pt' and no saved Arch4 LORMA "
                "checkpoint with architecture_config were found. Attach the Phase-1 Arch4 dataset."
            )
        lorma_ck = torch.load(lorma_path, map_location="cpu", weights_only=False)
        cfg = lorma_ck.get("architecture_config") if isinstance(lorma_ck, dict) else None
        if not isinstance(cfg, dict):
            raise RuntimeError(
                f"LORMA checkpoint has no dict 'architecture_config': {lorma_path}. "
                "Attach the original Phase-1 Arch4 checkpoint instead."
            )
        cfg = dict(cfg)
        # Accommodate wrappers that store the actual config one level down.
        if "cfg" in cfg and isinstance(cfg["cfg"], dict):
            cfg = dict(cfg["cfg"])
        source = f"fallback architecture_config from LORMA checkpoint: {lorma_path}"
        del lorma_ck

    missing = [k for k in ARCH4_REQUIRED_CFG_KEYS if k not in cfg]
    if missing:
        raise RuntimeError(
            "Arch4 configuration is missing required keys: " + ", ".join(missing) +
            f"\nConfig source: {source}\nConfig keys: {sorted(cfg.keys())}\n"
            "Attach the original Phase-1 Arch4 checkpoint; do not guess architecture dimensions."
        )

    print("Arch4 configuration source:", source)
    print("Arch4 config summary:", {
        k: cfg[k] for k in ARCH4_REQUIRED_CFG_KEYS
    })
    return cfg, source


def load_arch4_fullft():
    cfg, cfg_source = resolve_arch4_config()
    ck_path = find_arch4_fullft_checkpoint()
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or "backbone" not in ck or "bio_head" not in ck:
        raise RuntimeError(f"Unexpected Phase-2 Arch4 baseline checkpoint format: {ck_path}")

    ns_model = Architecture4(cfg)
    # Strict loading ensures the fallback config reproduces the saved architecture.
    ns_model.load_state_dict(ck["backbone"], strict=True)
    head = nn.Linear(cfg["D_MODEL"], 25)
    head.load_state_dict(ck["bio_head"], strict=True)
    model = ns_model.to(DEVICE).eval()
    head = head.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    for p in head.parameters(): p.requires_grad_(False)
    print("Arch 4 config:", cfg_source)
    print("Arch 4 Phase-2 full fine-tuning checkpoint:", ck_path)
    print("Stored validation F1:", ck["metrics"]["strict_macro_f1"])
    print("Epoch:", ck["epoch"])
    return model, head, ck, ck_path


model4, head4, ck4, ck4_path = load_arch4_fullft()
tag2id4 = ck4.get("tag2id", data.tag2id)
tag_fn4 = get_custom_tag_fn(model4.hidden_states, head4, tag2id4)
metrics4 = R.evaluate(data, tag_fn4)
stored_f1_4 = float(ck4["metrics"]["strict_macro_f1"])
actual_f1_4 = float(metrics4["strict_macro_f1"])
print("Arch 4 full-FT strict macro-F1 stored / re-evaluated:", stored_f1_4, actual_f1_4)
assert abs(actual_f1_4 - stored_f1_4) < 5e-3, "Arch 4 full-FT checkpoint did not reproduce its saved validation score."

name4 = unique_run_name("arch4_fullft_existing")
zip4 = save_test_submission(name4, data, tag_fn4)
result4 = {
    "name": name4, "arch": "arch4", "method": "existing Phase-2 full fine-tuning checkpoint",
    "best_epoch": int(ck4["epoch"]), "best": metrics4,
    "trainable_params": int(ck4.get("n_params", 0)), "submission_zip": zip4,
    "peak_gpu_gb": None, "train_seconds": None,
    "source_checkpoint": str(ck4_path), "evaluator": "custom validation evaluator",
}
print("Arch 4 full-FT submission:", zip4)
result4["artifact_dir"] = str(archive_experiment_artifacts(
    arch="arch4", method="Full Fine-Tuning (existing Phase-2 checkpoint)",
    run_name=name4, result=result4, submission_zip=zip4,
    source_checkpoint=ck4_path, copy_source_checkpoint=False,
    note="Existing Phase-2 full-fine-tuned checkpoint; source checkpoint referenced, not copied because it is large."
))

# Cache Arch 4 logits for the same 1024-token training windows, with BIO class reordering if needed.
teacher_cache_arch4 = cache_logits(
    model4.hidden_states, head4, data,
    label_source_map=tag2id4,
    label="Arch 4 full-FT",
)
print("Arch 4 teacher-cache shape:", tuple(teacher_cache_arch4.shape))

# Read the pre-existing LORMA score for the experiment log. We do not silently guess its adapter forward method.
lorma_path = find_unique_file("best_lorma_strict_macro_f1.pt", contains="phase2_arch4_lorma", required=False)
existing_lorma_summary = None
if lorma_path is not None:
    lorma_ck = torch.load(lorma_path, map_location="cpu", weights_only=False)
    lm = lorma_ck.get("metrics", {})
    existing_lorma_summary = {
        "checkpoint": str(lorma_path),
        "epoch": lorma_ck.get("epoch"),
        "strict_macro_f1": lm.get("strict_macro_f1"),
        "strict_macro_precision": lm.get("strict_macro_precision"),
        "strict_macro_recall": lm.get("strict_macro_recall"),
        "trainable_parameters": lorma_ck.get("trainable_parameters"),
        "config": lorma_ck.get("config"),
        "note": "Existing saved LORMA result recorded for analysis; its inference wrapper is not inferred from state-dict names.",
    }
    print("Existing Arch 4 LORMA saved F1:", existing_lorma_summary["strict_macro_f1"])
    del lorma_ck
    existing_lorma_summary["artifact_dir"] = str(archive_experiment_artifacts(
        arch="arch4", method="LORMA + identity inflation (existing checkpoint)",
        run_name="arch4_lorma_existing", result=existing_lorma_summary,
        submission_zip=None, source_checkpoint=lorma_path,
        copy_source_checkpoint=True,
        note=("Existing saved LORMA validation result. This notebook does not reconstruct its inference wrapper, "
              "so no prediction.json or submission.zip is fabricated for LORMA.")
    ))


Arch4 configuration source: Phase-1 checkpoint config: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
Arch4 config summary: {'VOCAB_SIZE': 40000, 'D_MODEL': 1024, 'N_LAYERS': 12, 'N_Q_HEADS': 16, 'N_KV_HEADS': 8, 'D_FF': 3328, 'WINDOW': 256, 'GLOBAL_EVERY': 3, 'DROPOUT': 0.0, 'MAX_SEQ_LEN': 1024, 'SEQ_LEN': 1024, 'ROPE_BASE': 10000.0, 'INIT_STD': 0.02}
Arch 4 config: Phase-1 checkpoint config: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
Arch 4 Phase-2 full fine-tuning checkpoint: /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/best_strict_macro_f1.pt
Stored validation F1: 0.3241674447932627
Epoch: 4
Arch 4 full-FT strict macro-F1 stored / re-evaluated: 0.3241674447932627 0.3241674447932627
Submission ZIP created: /kaggle/working/sub_arch4_fullft_existing_20261011_073309_89638/submission.zip
Arch 4 full-FT submission: /kaggle/working/sub

## 8. Experiment D/E — Arch 1 LoRA hard-label control and multi-teacher KD

In [9]:
# Free the Arch 4 GPU model only after caching its logits and generating its submission.
del model4, head4, ck4, tag_fn4
fresh_gpu()

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_LR = 1e-4
HEAD_LR = 2e-4
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.30
TEACHER4_WEIGHT = 0.70
TEACHER3_WEIGHT = 0.30

class StudentLoRALinear(nn.Module):
    """Frozen linear base + additive low-rank update, initialized as an identity update."""
    def __init__(self, base_layer, rank, alpha, dropout):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError(f"Expected nn.Linear, got {type(base_layer).__name__}")
        self.base = base_layer
        for p in self.base.parameters(): p.requires_grad_(False)
        self.rank = int(rank)
        self.scaling = float(alpha) / self.rank
        self.lora_dropout = nn.Dropout(dropout)
        device, dtype = base_layer.weight.device, base_layer.weight.dtype
        self.lora_A = nn.Linear(base_layer.in_features, self.rank, bias=False, device=device, dtype=dtype)
        self.lora_B = nn.Linear(self.rank, base_layer.out_features, bias=False, device=device, dtype=dtype)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
    def forward(self, x):
        return self.base(x) + self.scaling * self.lora_B(self.lora_A(self.lora_dropout(x)))


def make_student(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    bb = load_backbone("arch1", device=DEVICE)
    for p in bb.model.parameters(): p.requires_grad_(False)
    probe_ids = data.train.input_ids[:1].to(DEVICE)
    with torch.inference_mode():
        before = bb.hidden(probe_ids).float().cpu()
    targets = [("attn", "q_proj"), ("attn", "k_proj"), ("attn", "v_proj"), ("attn", "out_proj"), ("mlp", "fc1"), ("mlp", "fc2")]
    count = 0
    for block in bb.model.blocks:
        for parent_name, attr in targets:
            parent = getattr(block, parent_name)
            base = getattr(parent, attr)
            if not isinstance(base, nn.Linear):
                raise TypeError(f"Arch1 target {parent_name}.{attr} is not nn.Linear")
            setattr(parent, attr, StudentLoRALinear(base, LORA_RANK, LORA_ALPHA, LORA_DROPOUT))
            count += 1
    assert count == 48, f"Expected 48 Arch1 LoRA targets, got {count}"
    lora_params = []
    for name, p in bb.model.named_parameters():
        if ".lora_A." in name or ".lora_B." in name:
            p.requires_grad_(True)
            lora_params.append(p)
        else:
            p.requires_grad_(False)
    head = nn.Linear(bb.d_model, 25, device=next(bb.model.parameters()).device, dtype=next(bb.model.parameters()).dtype)
    for p in head.parameters(): p.requires_grad_(True)
    with torch.inference_mode():
        after = bb.hidden(probe_ids).float().cpu()
    diff = (before - after).abs().max().item()
    assert diff < 1e-5, f"Arch1 LoRA identity check failed: {diff}"
    optimizer = torch.optim.AdamW(
        [{"params": lora_params, "lr": LORA_LR}, {"params": list(head.parameters()), "lr": HEAD_LR}],
        weight_decay=0.01,
    )
    print("Fresh Arch1 LoRA modules:", count, "| trainable LoRA parameters:", f"{sum(p.numel() for p in lora_params):,}", "| identity diff:", diff)
    return bb, head, optimizer


def distillation_loss_factory(cache3, cache4, temperature=KD_TEMPERATURE, alpha=KD_ALPHA, w4=TEACHER4_WEIGHT, w3=TEACHER3_WEIGHT):
    @torch.no_grad()
    def blended_teacher(idx, device):
        # idx indexes rows in the exact training-window order used to cache teacher logits.
        l3 = cache3[idx.detach().cpu()].to(device=device, dtype=torch.float32)
        l4 = cache4[idx.detach().cpu()].to(device=device, dtype=torch.float32)
        p3 = F.softmax(l3 / temperature, dim=-1)
        p4 = F.softmax(l4 / temperature, dim=-1)
        return w3 * p3 + w4 * p4

    def loss_fn(logits, labels, idx):
        flat_logits = logits.reshape(-1, 25).float()
        flat_labels = labels.reshape(-1)
        hard = F.cross_entropy(flat_logits, flat_labels, ignore_index=-100)
        mask = flat_labels.ne(-100)
        if not bool(mask.any()):
            return hard
        # Keep all T token positions until both tensors are aligned.
        # Apply the valid-token mask exactly once to each tensor.
        teacher_probs = blended_teacher(idx, logits.device).reshape(-1, 25)
        if teacher_probs.shape != flat_logits.shape:
            raise RuntimeError(
                f"Teacher/student shape mismatch: teacher={tuple(teacher_probs.shape)}, "
                f"student={tuple(flat_logits.shape)}"
            )
        student_logp = F.log_softmax(flat_logits[mask] / temperature, dim=-1)
        valid_teacher_probs = teacher_probs[mask]
        if student_logp.shape != valid_teacher_probs.shape:
            raise RuntimeError(
                f"Masked KD shape mismatch: student={tuple(student_logp.shape)}, "
                f"teacher={tuple(valid_teacher_probs.shape)}"
            )
        soft = F.kl_div(student_logp, valid_teacher_probs, reduction="batchmean") * (temperature ** 2)
        total = (1.0 - alpha) * hard + alpha * soft
        if not torch.isfinite(total):
            raise RuntimeError("Non-finite KD loss")
        return total
    return loss_fn



# Fail fast on KD mask/alignment bugs before launching a multi-epoch run.
_kd_probe_loss_fn = distillation_loss_factory(teacher_cache_arch3, teacher_cache_arch4)
_kd_probe_logits = torch.randn(
    1, data.max_len, 25, device=DEVICE, dtype=torch.float32, requires_grad=True
)
_kd_probe_labels = data.train.labels[:1].to(DEVICE)
_kd_probe_idx = torch.tensor([0], dtype=torch.long)
_kd_probe_loss = _kd_probe_loss_fn(
    _kd_probe_logits, _kd_probe_labels, _kd_probe_idx
)
assert _kd_probe_loss.ndim == 0 and torch.isfinite(_kd_probe_loss), "KD loss smoke test failed"
_kd_probe_loss.backward()
assert _kd_probe_logits.grad is not None and torch.isfinite(_kd_probe_logits.grad).all(), \
    "KD gradient smoke test failed"
print("KD loss smoke test: PASS | loss:", round(float(_kd_probe_loss.detach()), 6))
del _kd_probe_loss_fn, _kd_probe_logits, _kd_probe_labels, _kd_probe_idx, _kd_probe_loss

# Hard-label control (same architecture, seed, windows, epochs, optimizer rates).
bb1_hard, head1_hard, optimizer1_hard = make_student(SEED)
modules1_hard = {"backbone": bb1_hard.model, "head": head1_hard}
name1_hard, result1_hard = run_candidate(
    "arch1_lora_hard_control", "arch1", "LoRA rank=8 hard-label control", modules1_hard,
    bb1_hard.hidden, head1_hard, optimizer1_hard, data, epochs=5,
    notes="Matched hard-label control for multi-teacher KD; Arch1 LoRA r=8 alpha=16 dropout=0.05; same seed/windows/optimizer/epochs.",
)
print("Arch1 LoRA hard-control best F1:", result1_hard["best"]["strict_macro_f1"])
del bb1_hard, head1_hard, optimizer1_hard, modules1_hard
fresh_gpu()

# Multi-teacher KD: 70% Arch4 and 30% Arch3 teacher probabilities, plus gold CE.
bb1_kd, head1_kd, optimizer1_kd = make_student(SEED)
modules1_kd = {"backbone": bb1_kd.model, "head": head1_kd}
kd_loss = distillation_loss_factory(teacher_cache_arch3, teacher_cache_arch4)
name1_kd, result1_kd = run_candidate(
    "arch1_lora_kd_arch3_arch4_maskfix", "arch1", "LoRA rank=8 + multi-teacher KD (mask fix)", modules1_kd,
    bb1_kd.hidden, head1_kd, optimizer1_kd, data, epochs=5,
    notes="Arch1 LoRA student; soft targets are 0.30*Arch3-Houlsby + 0.70*Arch4-full-FT teacher probabilities; T=2; KD alpha=0.30; hard CE remains included.",
    loss_fn=kd_loss,
)
print("Arch1 multi-teacher KD best F1:", result1_kd["best"]["strict_macro_f1"])
print("Arch1 KD submission:", result1_kd["submission_zip"])
del bb1_kd, head1_kd, optimizer1_kd, modules1_kd, kd_loss
fresh_gpu()

KD loss smoke test: PASS | loss: 5.787005
Fresh Arch1 LoRA modules: 48 | trainable LoRA parameters: 884,736 | identity diff: 0.0
Trainable parameters: 903,961 | LR groups: [0.0001, 0.0002]
[arch1_lora_hard_control_20261011_073309_89638] trainable 903,961 = 1.015% of backbone (89,095,680); all modules 89,114,905
  epoch 1/5 batch 1/502 loss 3.5748
  epoch 1/5 batch 50/502 loss 2.3727
  epoch 1/5 batch 100/502 loss 1.3755
  epoch 1/5 batch 150/502 loss 0.8400
  epoch 1/5 batch 200/502 loss 0.6182
  epoch 1/5 batch 250/502 loss 1.0229
  epoch 1/5 batch 300/502 loss 0.9591
  epoch 1/5 batch 350/502 loss 0.2696
  epoch 1/5 batch 400/502 loss 0.3103
  epoch 1/5 batch 450/502 loss 0.5763
  epoch 1/5 batch 500/502 loss 0.3183
  epoch 1/5 batch 502/502 loss 0.4024
[arch1_lora_hard_control_20261011_073309_89638] epoch 1: loss 0.9678 | P 0.0000 R 0.0000 F1 0.0000  (custom validation)
  new best saved
  epoch 2/5 batch 1/502 loss 0.1317
  epoch 2/5 batch 50/502 loss 0.6478
  epoch 2/5 batch 100/50

## 9. Select one candidate per Phase 1 architecture and validate ZIPs

In [10]:
# Candidate scores are on the same custom 14-document validation split except Arch2, which preserves its 768/512 windows.
# The final four keep one model from each Phase 1 architecture.
arch1_candidates = [
    ("Arch1 BitFit", result1_bit),
    ("Arch1 LoRA hard control", result1_hard),
    ("Arch1 LoRA multi-teacher KD", result1_kd),
]
arch1_label, arch1_best = max(arch1_candidates, key=lambda x: float(x[1]["best"]["strict_macro_f1"]))

final_candidates = {
    "arch1": {"label": arch1_label, "result": arch1_best},
    "arch2": {"label": "Arch2 BitFit", "result": result2},
    "arch3": {"label": "Arch3 Houlsby", "result": result3},
    "arch4": {"label": "Arch4 full fine-tuning (existing checkpoint)", "result": result4},
}

FINAL_DIR = WORK_DIR / "final_submissions"
FINAL_DIR.mkdir(parents=True, exist_ok=True)


def check_zip(path, dataset):
    path = Path(path)
    assert path.is_file(), f"Missing submission ZIP: {path}"
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        assert names == ["prediction.json"], f"ZIP must contain only root prediction.json, got {names}"
        payload = json.loads(zf.read("prediction.json").decode("utf-8"))
    expected_ids = {str(k) for k in dataset.test_text}
    assert set(payload) == expected_ids, f"Test ID mismatch: missing={len(expected_ids-set(payload))}, extra={len(set(payload)-expected_ids)}"
    labels = set(dataset.entity_labels)
    for doc_id, spans in payload.items():
        orig_id = doc_id if doc_id in dataset.test_text else next(k for k in dataset.test_text if str(k) == doc_id)
        text = dataset.test_text[orig_id]
        assert isinstance(spans, list)
        for span in spans:
            _normalise_prediction_span(span, text, doc_id, labels)
    return {"path": str(path), "test_ids": len(payload), "predicted_spans": sum(len(v) for v in payload.values()), "zip_entries": names}


final_manifest = {
    "run_id": RUN_ID,
    "artifact_root": str(ARTIFACT_ROOT),
    "artifact_folder_policy": "Each completed experiment is saved under a human-readable Arch + technique folder.",
    "test_documents": len(data.test_text),
    "oracle_macro_f1_1024": float(oracle_main["strict_macro_f1"]),
    "oracle_macro_f1_768": float(oracle_bitfit["strict_macro_f1"]),
    "arch1_candidates": [
        {"label": label, "run": res["name"], "strict_macro_f1": float(res["best"]["strict_macro_f1"])}
        for label, res in arch1_candidates
    ],
    "existing_lorma": existing_lorma_summary,
    "final_candidates": {},
}

for arch, item in final_candidates.items():
    res = item["result"]
    src_zip = res.get("submission_zip")
    if not src_zip:
        raise FileNotFoundError(f"Candidate {arch} has no ZIP")
    dst_zip = FINAL_DIR / f"{arch}_best_submission.zip"
    shutil.copy2(src_zip, dst_zip)
    zip_check = check_zip(dst_zip, data)
    item["final_zip"] = str(dst_zip)
    item["zip_check"] = zip_check
    item["strict_macro_f1"] = float(res["best"]["strict_macro_f1"] if "best" in res else res["best"]["strict_macro_f1"])
    final_manifest["final_candidates"][arch] = {
        "label": item["label"],
        "run": res["name"],
        "strict_macro_f1": item["strict_macro_f1"],
        "submission_zip": str(dst_zip),
        "zip_check": zip_check,
    }

# The overall highest-validation candidate is also copied to the conventional competition filename.
overall_arch, overall_item = max(final_candidates.items(), key=lambda kv: kv[1]["strict_macro_f1"])
shutil.copy2(overall_item["final_zip"], WORK_DIR / "submission.zip")
final_manifest["overall_best_validation_candidate"] = {
    "architecture": overall_arch,
    "label": overall_item["label"],
    "strict_macro_f1": overall_item["strict_macro_f1"],
    "submission_zip": str(WORK_DIR / "submission.zip"),
}

# Produce a comparison CSV from all model candidates.
summary_rows = []
def add_summary(label, result, arch, source="trained in this notebook"):
    m = result["best"]
    summary_rows.append({
        "candidate": label,
        "arch": arch,
        "method": result.get("method", label),
        "strict_macro_f1": float(m["strict_macro_f1"]),
        "strict_macro_precision": float(m["strict_macro_precision"]),
        "strict_macro_recall": float(m["strict_macro_recall"]),
        "best_epoch": result.get("best_epoch"),
        "trainable_params": result.get("trainable_params"),
        "peak_gpu_gb": result.get("peak_gpu_gb"),
        "train_seconds": result.get("train_seconds"),
        "source": source,
        "submission_zip": result.get("submission_zip"),
        "artifact_dir": result.get("artifact_dir"),
    })

add_summary("Arch1 BitFit", result1_bit, "arch1")
add_summary("Arch1 LoRA hard control", result1_hard, "arch1")
add_summary("Arch1 LoRA multi-teacher KD", result1_kd, "arch1")
add_summary("Arch2 BitFit", result2, "arch2")
add_summary("Arch3 Houlsby", result3, "arch3")
summary_rows.append({
    "candidate": "Arch4 full fine-tuning existing checkpoint", "arch": "arch4", "method": "full fine-tuning",
    "strict_macro_f1": float(metrics4["strict_macro_f1"]),
    "strict_macro_precision": float(metrics4["strict_macro_precision"]),
    "strict_macro_recall": float(metrics4["strict_macro_recall"]),
    "best_epoch": int(result4["best_epoch"]),
    "trainable_params": None, "peak_gpu_gb": None, "train_seconds": None,
    "source": str(ck4_path), "submission_zip": zip4,
    "artifact_dir": result4.get("artifact_dir"),
})
if existing_lorma_summary:
    lm = existing_lorma_summary
    summary_rows.append({
        "candidate": "Arch4 LORMA existing checkpoint", "arch": "arch4", "method": "LORMA+ identity inflation",
        "strict_macro_f1": lm.get("strict_macro_f1"),
        "strict_macro_precision": lm.get("strict_macro_precision"),
        "strict_macro_recall": lm.get("strict_macro_recall"),
        "best_epoch": lm.get("epoch"), "trainable_params": lm.get("trainable_parameters"),
        "peak_gpu_gb": None, "train_seconds": None,
        "source": lm.get("checkpoint"), "submission_zip": None,
        "artifact_dir": lm.get("artifact_dir"),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("strict_macro_f1", ascending=False, na_position="last")
summary_path = WORK_DIR / "phase2_end_to_end_comparison.csv"
summary_df.to_csv(summary_path, index=False)
manifest_path = FINAL_DIR / "manifest.json"
manifest_path.write_text(json.dumps(final_manifest, indent=2), encoding="utf-8")
print(summary_df.to_string(index=False))
print("\nComparison CSV:", summary_path)
print("Manifest:", manifest_path)
print("Final ZIP directory:", FINAL_DIR)
print("Overall best-validation submission:", WORK_DIR / "submission.zip")

# Keep the comparison and selection manifest alongside all per-experiment folders.
shutil.copy2(summary_path, ARTIFACT_ROOT / "comparison.csv")
(ARTIFACT_ROOT / "manifest.json").write_text(
    json.dumps(final_manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)
print("Per-experiment artifact root:", ARTIFACT_ROOT)

                                 candidate  arch                                    method  strict_macro_f1  strict_macro_precision  strict_macro_recall  best_epoch  trainable_params  peak_gpu_gb  train_seconds                                                                                                    source                                                                             submission_zip                                                                     artifact_dir
Arch4 full fine-tuning existing checkpoint arch4                          full fine-tuning         0.324167                0.282767             0.393618           4               NaN          NaN            NaN    /kaggle/input/datasets/achiversx/phase2-arch4-nerupdates/phase2_arch4_baseline/best_strict_macro_f1.pt             /kaggle/working/sub_arch4_fullft_existing_20261011_073309_89638/submission.zip            /kaggle/working/phase2_architecture_results/Arch 4 + Full Fine-Tuning
           Arch4 LORMA

## 10. Run the official format validator on the final four ZIPs

In [11]:
validator_ok = {}
for arch, item in final_candidates.items():
    zpath = item["final_zip"]
    print("\n" + "=" * 72)
    print("OFFICIAL FORMAT VALIDATION:", arch, zpath)
    env = {**os.environ, "PRED": str(zpath), "TEXT": str(TEST_TEXT_PATH), "LABELS": str(LABELS_PATH)}
    proc = subprocess.run([sys.executable, str(VALIDATOR_PATH)], env=env, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr:
        print("STDERR:\n", proc.stderr[-4000:])
    validator_ok[arch] = (proc.returncode == 0)
    # Mirror the final official validation report into the corresponding experiment folder.
    artifact_dir = item["result"].get("artifact_dir")
    if artifact_dir:
        report_path = Path(artifact_dir) / "official_format_validator_final_candidate.json"
        report_path.write_text(json.dumps({
            "architecture": arch,
            "candidate": item["label"],
            "submission_zip": str(zpath),
            "returncode": int(proc.returncode),
            "passed": proc.returncode == 0,
            "stdout": proc.stdout,
            "stderr": proc.stderr[-6000:],
        }, indent=2), encoding="utf-8")
    if proc.returncode != 0:
        raise RuntimeError(f"Official validator failed for {arch}: exit code {proc.returncode}")

final_manifest["official_validator"] = validator_ok
manifest_path.write_text(json.dumps(final_manifest, indent=2, default=str), encoding="utf-8")
# Update comparison/manifest within the artifact root after final selection.
shutil.copy2(summary_path, ARTIFACT_ROOT / "comparison.csv")
(ARTIFACT_ROOT / "manifest.json").write_text(
    json.dumps(final_manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

# Bundle all per-experiment folders so one download contains every candidate's outputs.
if ARTIFACT_BUNDLE_PATH.exists():
    ARTIFACT_BUNDLE_PATH.unlink()
with zipfile.ZipFile(ARTIFACT_BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file():
            bundle.write(path, arcname=f"phase2_architecture_results/{path.relative_to(ARTIFACT_ROOT)}")

print("\nAll final ZIPs passed official format validation.")
print("Final files:")
for p in sorted(FINAL_DIR.iterdir()):
    print(" ", p, f"({p.stat().st_size / 1024**2:.2f} MB)")
print("\nPer-experiment folders:", ARTIFACT_ROOT)
for path in sorted(ARTIFACT_ROOT.iterdir()):
    if path.is_dir() and not path.name.startswith("_"):
        print(" ", path.name, "->", [q.name for q in sorted(path.iterdir()) if q.is_file()])
print("Artifact bundle ZIP:", ARTIFACT_BUNDLE_PATH)
print("\nNOTE: the validator checks submission format and IDs; it does not reveal private test F1.")



OFFICIAL FORMAT VALIDATION: arch1 /kaggle/working/final_submissions/arch1_best_submission.zip
documents in prediction: 35 | entities: 6363 | documents with no predicted entity: 1
predicted entities per label: {'PREC': 1571, 'STAT': 1302, 'DATE': 1168, 'CASENO': 1116, 'COURT': 744, 'AUTH': 270, 'JUDGE': 160, 'APP': 21, 'R.COUNSEL': 7, 'RESP': 3, 'A.COUNSEL': 1}

VALID: safe to upload


OFFICIAL FORMAT VALIDATION: arch2 /kaggle/working/final_submissions/arch2_best_submission.zip
documents in prediction: 35 | entities: 8178 | documents with no predicted entity: 1
predicted entities per label: {'PREC': 2319, 'DATE': 2143, 'STAT': 1214, 'CASENO': 1134, 'COURT': 868, 'AUTH': 357, 'JUDGE': 50, 'APP': 36, 'RESP': 33, 'WIT': 17, 'A.COUNSEL': 4, 'R.COUNSEL': 3}

VALID: safe to upload


OFFICIAL FORMAT VALIDATION: arch3 /kaggle/working/final_submissions/arch3_best_submission.zip
documents in prediction: 35 | entities: 5463 | documents with no predicted entity: 1
predicted entities per label: {'P

## Output files

After a successful run, Kaggle will contain one human-readable folder per completed experiment under `/kaggle/working/phase2_architecture_results/`. Each trained/inferred candidate folder contains `prediction.json`, `submission.zip`, `results.json`, `experiment_result.json`, `metrics.json`, metadata, and its trainable checkpoint when available. The historical LORMA folder contains its saved result/checkpoint reference; no LORMA prediction is created unless its inference wrapper is explicitly reconstructed and validated.

Example:

```text
phase2_architecture_results/
├── Arch 1 + BitFit/
│   ├── prediction.json
│   ├── submission.zip
│   ├── results.json
│   ├── experiment_result.json
│   ├── metrics.json
│   ├── best_trainable.pt
│   └── metadata.json
├── Arch 1 + LoRA (Hard-label Control)/
├── Arch 1 + LoRA + KD (Arch 3 + Arch 4)/
├── Arch 2 + BitFit/
├── Arch 3 + Houlsby/
├── Arch 4 + Full Fine-Tuning/
├── Arch 4 + LORMA/  # saved validation artifact; inference ZIP not fabricated
├── comparison.csv
└── manifest.json
```

A bundle of the entire organized directory is created at `/kaggle/working/phase2_architecture_results.zip`.

After a successful run, Kaggle will also contain:

- `final_submissions/arch1_best_submission.zip`
- `final_submissions/arch2_best_submission.zip`
- `final_submissions/arch3_best_submission.zip`
- `final_submissions/arch4_best_submission.zip`
- `final_submissions/manifest.json`
- `phase2_end_to_end_comparison.csv`
- `submission.zip` — highest custom-validation-F1 candidate across the four architectures

Every submission ZIP contains `prediction.json` directly at the archive root. Use the four architecture-specific ZIPs for the four-model submission package; use `submission.zip` when the platform requests a single file for one submission.

### Method references

- BitFit: Zaken et al., *BitFit: Simple Parameter-efficient Fine-tuning for Transformer-based Masked Language-models* (ACL 2022).
- LoRA: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021).
- Houlsby adapters: Houlsby et al., *Parameter-Efficient Transfer Learning for NLP* (ICML 2019).
- Knowledge distillation: Hinton et al., *Distilling the Knowledge in a Neural Network* (2015).
- LoRMA: Bihany et al., *LoRMA: Low-Rank Multiplicative Adaptation for LLMs* (Findings of ACL 2025).